[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maipatatohoon/bioinsight-360/blob/main/BioInsight_360_Colab_Runbook.ipynb)
[![Repository](https://img.shields.io/badge/source-bioinsight--360-181717?logo=github)](https://github.com/Maipatatohoon/bioinsight-360)
[**Download .ipynb**](https://raw.githubusercontent.com/Maipatatohoon/bioinsight-360/main/BioInsight_360_Colab_Runbook.ipynb)

> This notebook lives in **your repository** (`Maipatatohoon/bioinsight-360`, branch `main`).
> Open it with the badge above so you always get the latest version, and use
> **File → Save a copy in GitHub** if you change it and want the change pushed back.

# BioInsight 360 — Live Backend Runbook (v3)

This notebook installs **R + Bioconductor**, installs the **Python backend**, starts the **FastAPI server**, opens a **public HTTPS URL**, starts the **analysis worker**, and then runs a **real end-to-end job** to prove the whole pipeline works.

> ## The one thing you need from this notebook
> **STEP 11 prints a URL like `https://random-words.trycloudflare.com`.**
> Copy it and paste it into your website — STEP 14 explains exactly where.

| | |
|---|---|
| First run | **25–45 min** (mostly R package compilation) |
| Later runs, same runtime | ~2 min |
| GPU | **not needed** — use a CPU runtime |
| Keep the tab open | **yes**. If Colab disconnects, the URL dies. Re-run STEP 10 → 12. |

### How to use this notebook

1. `Runtime → Change runtime type → CPU` (GPU is not used).
2. Run the cells **in order**, top to bottom (▶ or `Ctrl+Enter`).
3. Do **not** use *Run all*: you want to see each step's result and stop at the first ❌.
4. At **STEP 11**, copy the URL immediately. You can re-print it any time with **STEP 14**.
5. If STEP 1 cannot connect, see the box below before changing anything in the notebook.

If STEP 1 says **"Failed to fetch"**, stop and read the box below — it is a browser/network issue, not the notebook.

## ⚠️ STEP 1 says "Failed to fetch"? Read this first

STEP 1 only imports `os`, `platform`, `sys` — it makes **no network calls**. The
message is Colab's browser↔runtime link failing, not this notebook.

1. **Allow third-party cookies for Colab.**
   - Firefox: `about:preferences#privacy` → Enhanced Tracking Protection → **Standard**;
     Manage Exceptions → allow `https://colab.research.google.com` and
     `https://*.googleusercontent.com`; restart Firefox.
   - Chrome: Settings → Privacy and security → Third-party cookies → **Allow third-party
     cookies**; add `colab.research.google.com` to the allowed list.
2. **Turn off content blockers** for Colab (uBlock Origin, Privacy Badger, Ghostery,
   AdGuard), then try a **new private window** and a blank notebook: `print("hi")`.
3. **Change network.** Campus/corporate Wi-Fi often blocks the WebSocket Colab needs.
   Connect the laptop to a **phone hotspot**, open Colab, and run STEP 1. The tunnel
   runs on Colab's VM, so after STEP 11 prints the URL you can switch the laptop back
   to Wi-Fi — the URL stays public. Starting Colab in a phone browser works too.

If `print("hi")` in a brand-new notebook also says "Failed to fetch", the problem is
the network/browser; no notebook change can fix it.

## Step map — what each cell does

| Step | What it does | Time | Must pass? |
|---|---|---|---|
| 1 | Configuration | 1 s | ✅ |
| 2 | System packages + R | 2–4 min | ✅ |
| 3 | BiocManager + `argparse` | 1 min | ✅ |
| 4 | R package **DESeq2** | **10–25 min** | ✅ |
| 5 | R package **edgeR** | 1–3 min | ✅ |
| 6 | R package **limma** | 1 min | ✅ |
| 7 | Python packages | 1 min | ✅ |
| 8 | Download backend code | 10 s | ✅ |
| 9 | Pre-flight check (all installs) | 5 s | ✅ |
| 10 | Start the backend | 10 s | ✅ |
| 11 | **Open the public URL** | 10–30 s | ✅ |
| 12 | Start the worker | 10 s | ✅ |
| 13 | End-to-end test job | 1–3 min | ✅ |
| 14 | **Re-print your URL** | 1 s | — |
| 15 | Logs / diagnostics | 1 s | — |
| 16 | *Optional:* thesis validation outputs + ZIP | 1–3 min | — |
| 17 | Stop / restart helpers + troubleshooting | — | — |

**If a step prints ❌, stop and fix that step before continuing.** The troubleshooting table is at the end.

In [ ]:
#@title STEP 1 — Configuration (you normally don't need to change anything) { display-mode: "form" }

BACKEND_REPO  = "https://github.com/Maipatatohoon/bioinsight-backend.git"
FRONTEND_REPO = "Maipatatohoon/bioinsight-360"   # used only for the demo dataset
FRONTEND_BRANCH = "main"

WORKDIR  = "/content/bioinsight-backend"          # backend code lives here
DATA_DIR = "/content/bioinsight_data"             # demo counts + metadata
DB_PATH  = "/content/bioinsight_jobs.db"          # job database
URL_FILE = "/content/BACKEND_URL.txt"             # STEP 11 writes the public URL here

BACKEND_LOG = "/content/backend.log"
WORKER_LOG  = "/content/worker.log"
TUNNEL_LOG  = "/content/cloudflared.log"

WORKER_TOKEN = "dev-token-123"   # must match on backend + worker

# Where your website runs — only used to print convenient links
SITE_URL       = "https://bioinsight-360.vercel.app"
LOCAL_SITE_URL = "http://localhost:3000"

# Set this only if you already have a public backend URL you want to reuse.
MANUAL_BACKEND_URL = ""

import os, platform, sys
os.makedirs(DATA_DIR, exist_ok=True)
print("Python  :", sys.version.split()[0])
print("Machine :", platform.platform())
print("CPUs    :", os.cpu_count())
print("Workdir :", WORKDIR)
print("DB      :", DB_PATH)

## STEP 2 — System packages and R

Installs R 4.x from the CRAN repository (instead of the older Ubuntu copy), plus the C libraries that Bioconductor packages compile against.

If the CRAN repository is unavailable for the runtime's Ubuntu release, the cell falls back to the distro's `r-base` automatically.

In [ ]:
%%bash
set -uo pipefail
export DEBIAN_FRONTEND=noninteractive
. /etc/os-release
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME:-jammy}}"
echo "Ubuntu codename: $CODENAME"

echo ">>> [1/7] apt update"
apt-get update -qq || true

echo ">>> [2/7] base build tools"
apt-get install -y -qq wget ca-certificates gnupg dirmngr software-properties-common \
    build-essential gfortran >/dev/null

echo ">>> [3/7] CRAN apt repository (R 4.x)"
wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc \
  | tee /usr/share/keyrings/cran.gpg >/dev/null
echo "deb [signed-by=/usr/share/keyrings/cran.gpg] https://cloud.r-project.org/bin/linux/ubuntu ${CODENAME}-cran40/" \
  > /etc/apt/sources.list.d/cran.list
if ! apt-get update -qq; then
  echo "    CRAN repository unavailable for ${CODENAME}; falling back to the distro R"
  rm -f /etc/apt/sources.list.d/cran.list
  apt-get update -qq || true
fi

echo ">>> [4/7] r-base + r-base-dev"
apt-get install -y -qq r-base r-base-dev >/dev/null || apt-get install -y -qq r-base >/dev/null

echo ">>> [5/7] system libraries for Bioconductor compilation"
for p in libcurl4-openssl-dev libssl-dev libxml2-dev libglpk-dev \
         libpng-dev libjpeg-dev libtiff-dev libfontconfig1-dev \
         libfreetype6-dev libharfbuzz-dev libfribidi-dev; do
  echo "      - $p"
  apt-get install -y -qq "$p" >/dev/null || echo "        (optional — skipped)"
done

echo ">>> [6/7] versions"
R --version | head -1 || echo "!! R IS NOT INSTALLED"

echo ">>> [7/7] disk"
df -h /content | tail -1
echo "STEP 2 done."

## STEP 3 — BiocManager + argparse

`argparse` is easy to forget: **every R engine in this backend calls `library(argparse)`**, but BiocManager does not install it. Without this cell the engines fail with *"there is no package called 'argparse'"*.

In [ ]:
%%bash
echo ">>> Installing BiocManager (CRAN)"
Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); install.packages("BiocManager", repos = "https://cloud.r-project.org")'

echo ""
echo ">>> Installing argparse (required by all three R engines)"
Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); install.packages("argparse", repos = "https://cloud.r-project.org")'

echo ""
echo ">>> Bioconductor version selected for this R:"
Rscript -e 'cat(as.character(BiocManager::version()), "\n")'

echo ""
Rscript -e 'cat("BiocManager:", requireNamespace("BiocManager", quietly=TRUE), "| argparse:", requireNamespace("argparse", quietly=TRUE), "\n")'

## STEP 4 — DESeq2  *(the long one)*

This is usually **10–25 minutes** because Bioconductor compiles from source. Leave the tab open. When it finishes you should see `DESeq2 installed: TRUE`.

In [ ]:
%%bash
echo ">>> DESeq2 — compiling. This is the slow step; do not stop it."
time Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); BiocManager::install("DESeq2", ask = FALSE, update = FALSE)'
echo ""
Rscript -e 'cat("\nDESeq2 installed:", requireNamespace("DESeq2", quietly = TRUE), "\n")'

## STEP 5 — edgeR

In [ ]:
%%bash
echo ">>> edgeR"
time Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); BiocManager::install("edgeR", ask = FALSE, update = FALSE)'
echo ""
Rscript -e 'cat("\nedgeR installed:", requireNamespace("edgeR", quietly = TRUE), "\n")'

## STEP 6 — limma

In [ ]:
%%bash
echo ">>> limma"
time Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); BiocManager::install("limma", ask = FALSE, update = FALSE)'
echo ""
Rscript -e 'cat("\nlimma installed:", requireNamespace("limma", quietly = TRUE), "\n")'

## STEP 7 — Python packages, one by one

In [ ]:
# STEP 7 — Python packages (do NOT upgrade Colab's scientific stack)
import subprocess, sys, importlib

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", *args]
    print("$ pip install " + " ".join(args), flush=True)
    r = subprocess.run(cmd, text=True, capture_output=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-8:]
    for line in tail:
        print("   ", line)
    if r.returncode:
        print(f"   pip exit {r.returncode} — continuing; STEP 9 will verify")
    return r.returncode

# Only the packages Colab does not already ship.
pip_install("fastapi", "uvicorn", "python-multipart")

# Reuse Colab's pandas/numpy/scipy/sklearn/pydantic; install only if absent.
for mod, pkg in [("pandas", "pandas"), ("numpy", "numpy"), ("sklearn", "scikit-learn"),
                 ("scipy", "scipy"), ("httpx", "httpx"), ("requests", "requests"),
                 ("pydantic", "pydantic")]:
    try:
        importlib.import_module(mod)
        print(f"OK   {mod} already available")
    except Exception:
        pip_install(pkg)

print("\nImport check:")
import fastapi, uvicorn, pandas, numpy, pydantic, sklearn, scipy, httpx, requests, multipart
for m in (fastapi, uvicorn, pandas, numpy, pydantic, sklearn, scipy, httpx, requests):
    print(f"  {m.__name__:<12} {getattr(m, '__version__', 'ok')}")
print("  multipart    ok")


## STEP 8 — Download the backend code

In [ ]:
# STEP 8 — clone or update bioinsight-backend
import os, subprocess

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    print("Updating existing clone ...")
    subprocess.run(["git", "-C", WORKDIR, "fetch", "--depth", "1", "origin", "main"], check=False)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", "origin/main"], check=False)
else:
    print("Cloning", BACKEND_REPO)
    subprocess.run(["git", "clone", "--depth", "1", BACKEND_REPO, WORKDIR], check=True)

print(subprocess.run(
    ["git", "-C", WORKDIR, "log", "-1", "--format=%h %ad %s", "--date=short"],
    capture_output=True, text=True).stdout.strip())

required = [
    "app/main.py",
    "app/jobs/job_manager.py",
    "workers/colab_worker.py",
    "app/deseq2/deseq2_script.R",
    "app/edger/edger_script.R",
    "app/limma/limma_script.R",
]
missing = [f for f in required if not os.path.exists(os.path.join(WORKDIR, f))]
print("Required files:", "OK" if not missing else f"MISSING {missing}")

# --- compatibility hotfix: NA/Inf statistics must not 500 the results API ---
import base64
HOTFIX_B64 = "ZGlmZiAtLWdpdCBhL2FwcC9hbmFseXNpcy92YWxpZGF0b3IucHkgYi9hcHAvYW5hbHlzaXMvdmFsaWRhdG9yLnB5CmluZGV4IGIxMTdkNGQuLjY3MGZkOTMgMTAwNjQ0Ci0tLSBhL2FwcC9hbmFseXNpcy92YWxpZGF0b3IucHkKKysrIGIvYXBwL2FuYWx5c2lzL3ZhbGlkYXRvci5weQpAQCAtOTksNiArOTksMTQgQEAgZGVmIHZhbGlkYXRlX2FuYWx5c2lzX2lucHV0KGNvdW50c19kZjogcGQuRGF0YUZyYW1lLCBtZXRhX2RmOiBwZC5EYXRhRnJhbWUsIHBhcmEKICAgICAgICAgZWxzZToKICAgICAgICAgICAgIHJhaXNlIERhdGFWYWxpZGF0aW9uRXJyb3IoZiJDb25kaXRpb24gY29sdW1uICd7Y29uZGl0aW9uX2NvbH0nIG5vdCBmb3VuZCBpbiBtZXRhZGF0YSBjb2x1bW5zOiB7bGlzdChtZXRhX2RmLmNvbHVtbnMpfS4iKQogCisgICAgIyBUaGUgUiBlbmdpbmVzIGJ1aWxkIHRoZWlyIGRlc2lnbiBmb3JtdWxhIGFyb3VuZCBhIGNvbHVtbiBsaXRlcmFsbHkgbmFtZWQKKyAgICAjIGBjb25kaXRpb25gICh0aGUgLS1kZXNpZ24gZGVmYXVsdCBpbiBhcHAvKi8qLlIpLiBOb3JtYWxpc2Ugd2hpY2hldmVyCisgICAgIyBncm91cGluZyBjb2x1bW4gd2UgcmVzb2x2ZWQgc28gbWV0YWRhdGEgdXNpbmcgYGdyb3VwYCwgYHRyZWF0bWVudGAsIGV0Yy4KKyAgICAjIGRvZXMgbm90IGZhaWwgd2l0aCAib2JqZWN0ICdjb25kaXRpb24nIG5vdCBmb3VuZCIuCisgICAgaWYgY29uZGl0aW9uX2NvbCAhPSAiY29uZGl0aW9uIjoKKyAgICAgICAgbWV0YV9kZiA9IG1ldGFfZGYucmVuYW1lKGNvbHVtbnM9e2NvbmRpdGlvbl9jb2w6ICJjb25kaXRpb24ifSkKKyAgICAgICAgY29uZGl0aW9uX2NvbCA9ICJjb25kaXRpb24iCisKICAgICBjb3VudHNfcGVyX2dyb3VwID0gbWV0YV9kZltjb25kaXRpb25fY29sXS52YWx1ZV9jb3VudHMoKQogICAgIGZvciBncm91cF9uYW1lLCBjb3VudCBpbiBjb3VudHNfcGVyX2dyb3VwLml0ZW1zKCk6CiAgICAgICAgIGlmIGNvdW50IDwgMjoKZGlmZiAtLWdpdCBhL2FwcC9qb2JzL2pvYl9tYW5hZ2VyLnB5IGIvYXBwL2pvYnMvam9iX21hbmFnZXIucHkKaW5kZXggMWM3NDJiNi4uY2IxMDA1OCAxMDA2NDQKLS0tIGEvYXBwL2pvYnMvam9iX21hbmFnZXIucHkKKysrIGIvYXBwL2pvYnMvam9iX21hbmFnZXIucHkKQEAgLTExLDEyICsxMSw0OCBAQCBpbXBvcnQgc3FsaXRlMwogaW1wb3J0IHBhbmRhcyBhcyBwZAogaW1wb3J0IG51bXB5IGFzIG5wCiBpbXBvcnQgb3MKK2ltcG9ydCByZQogaW1wb3J0IGpzb24KIGZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBBbnksIE9wdGlvbmFsLCBMaXN0CiBmcm9tIGFwcC5hbmFseXNpcy52YWxpZGF0b3IgaW1wb3J0IHZhbGlkYXRlX2FuYWx5c2lzX2lucHV0LCBEYXRhVmFsaWRhdGlvbkVycm9yCiBmcm9tIGFwcC5hbmFseXNpcy5ydW5uZXIgaW1wb3J0IFJFbmdpbmVSdW5uZXIKIGZyb20gYXBwLmFuYWx5c2lzLnFjIGltcG9ydCBjb21wdXRlX3FjX21ldHJpY3MKIAorCitkZWYgX25vcm1hbGlzZV9kZXNpZ25fcGFyYW1zKHBhcmFtczogRGljdFtzdHIsIEFueV0sIG1ldGFfcGF0aDogc3RyKSAtPiBEaWN0W3N0ciwgQW55XToKKyAgICAiIiJNYWtlIGRlc2lnbi9jb250cmFzdCBtYXRjaCB0aGUgY2xlYW5lZCBtZXRhZGF0YSBjb2x1bW4gbmFtZXMuCisKKyAgICB2YWxpZGF0ZV9hbmFseXNpc19pbnB1dCgpIHJlbmFtZXMgd2hpY2hldmVyIGdyb3VwaW5nIGNvbHVtbiB3YXMgcmVzb2x2ZWQgdG8KKyAgICBgYGNvbmRpdGlvbmBgLiBDYWxsZXJzIHRoYXQgc3RpbGwgc2VuZCB0aGUgb3JpZ2luYWwgbmFtZSAoZS5nLiB0aGUgb2xkZXIKKyAgICBmcm9udGVuZCBzZW50IGBgfiBHcm91cGBgKSB3b3VsZCBvdGhlcndpc2UgZmFpbCB3aXRoCisgICAgIm9iamVjdCAnR3JvdXAnIG5vdCBmb3VuZCIuIFJld3JpdGUgdW5rbm93biBmYWN0b3IgbmFtZXMgdG8gYGBjb25kaXRpb25gYC4KKyAgICAiIiIKKyAgICBwYXJhbXMgPSBkaWN0KHBhcmFtcyBvciB7fSkKKyAgICB0cnk6CisgICAgICAgIGNvbHVtbnMgPSBsaXN0KHBkLnJlYWRfY3N2KG1ldGFfcGF0aCwgbnJvd3M9MCkuY29sdW1ucykKKyAgICBleGNlcHQgRXhjZXB0aW9uOgorICAgICAgICByZXR1cm4gcGFyYW1zCisgICAgaWYgImNvbmRpdGlvbiIgbm90IGluIGNvbHVtbnM6CisgICAgICAgIHJldHVybiBwYXJhbXMKKworICAgIGRlc2lnbiA9IHN0cihwYXJhbXMuZ2V0KCJkZXNpZ24iKSBvciAifiBjb25kaXRpb24iKQorICAgIHJocyA9IGRlc2lnbi5zcGxpdCgifiIsIDEpWzFdIGlmICJ+IiBpbiBkZXNpZ24gZWxzZSBkZXNpZ24KKyAgICByaHMgPSByZS5zdWIoCisgICAgICAgIHIiW0EtWmEtel8uXVtBLVphLXowLTlfLl0qIiwKKyAgICAgICAgbGFtYmRhIG06IG0uZ3JvdXAoMCkgaWYgbS5ncm91cCgwKSBpbiBjb2x1bW5zIGVsc2UgImNvbmRpdGlvbiIsCisgICAgICAgIHJocywKKyAgICApCisgICAgcGFyYW1zWyJkZXNpZ24iXSA9ICJ+ICIgKyByaHMuc3RyaXAoKQorICAgIHBhcmFtc1siY29uZGl0aW9uX2NvbCJdID0gImNvbmRpdGlvbiIKKworICAgIGNvbnRyYXN0ID0gcGFyYW1zLmdldCgiY29udHJhc3QiKQorICAgIGlmIGlzaW5zdGFuY2UoY29udHJhc3QsIHN0cikgYW5kIGNvbnRyYXN0OgorICAgICAgICBmYWN0b3IsIHNlcCwgcmVzdCA9IGNvbnRyYXN0LnBhcnRpdGlvbigiXyIpCisgICAgICAgIGlmIHNlcCBhbmQgZmFjdG9yIGFuZCBmYWN0b3Igbm90IGluIGNvbHVtbnM6CisgICAgICAgICAgICBwYXJhbXNbImNvbnRyYXN0Il0gPSAiY29uZGl0aW9uXyIgKyByZXN0CisgICAgcmV0dXJuIHBhcmFtcworCisKIGNsYXNzIEpvYlN0YXRlOgogICAgIFFVRVVFRCA9ICJRVUVVRUQiCiAgICAgVkFMSURBVElORyA9ICJWQUxJREFUSU5HIgpAQCAtNDksOCArODUsOCBAQCBjbGFzcyBKb2JTdG9yZToKICAgICAgICAgICAgIGNscy5faW5zdGFuY2UgPSBpbnN0CiAgICAgICAgIHJldHVybiBjbHMuX2luc3RhbmNlCiAKLSAgICBAc3RhdGljbWV0aG9kCi0gICAgZGVmIF9kZWZhdWx0X2RiX3BhdGgoKSAtPiBzdHI6CisgICAgQGNsYXNzbWV0aG9kCisgICAgZGVmIF9kZWZhdWx0X2RiX3BhdGgoY2xzKSAtPiBzdHI6CiAgICAgICAgICMgYXBwL2pvYnMvam9iX21hbmFnZXIucHkgLT4gYXBwL2pvYnMgLT4gYXBwIC0+IGJhY2tlbmQKICAgICAgICAgYmFja2VuZF9yb290ID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKQogICAgICAgICByZXR1cm4gb3MucGF0aC5qb2luKGJhY2tlbmRfcm9vdCwgImRhdGEiLCBjbHMuX0RFRkFVTFRfREJfRklMRU5BTUUpCkBAIC0yOTYsNyArMzMyLDcgQEAgY2xhc3MgTG9jYWxSdW50aW1lKENvbXB1dGVSdW50aW1lKToKICAgICAgICAgICAgICgiZWRnZXIiLCBvcy5wYXRoLmpvaW4oYXBwX2RpciwgImVkZ2VyIiwgImVkZ2VyX3NjcmlwdC5SIikpLAogICAgICAgICAgICAgKCJsaW1tYSIsIG9zLnBhdGguam9pbihhcHBfZGlyLCAibGltbWEiLCAibGltbWFfc2NyaXB0LlIiKSksCiAgICAgICAgIF0KLSAgICAgICAgcGFyYW1zID0gam9iLmdldCgicGFyYW1zIiwge30pIG9yIHt9CisgICAgICAgIHBhcmFtcyA9IF9ub3JtYWxpc2VfZGVzaWduX3BhcmFtcyhqb2IuZ2V0KCJwYXJhbXMiLCB7fSkgb3Ige30sIG1ldGFfcGF0aCkKICAgICAgICAgcmVzdWx0cyA9IHt9CiAgICAgICAgIHdhcm5pbmdzID0gW10KICAgICAgICAgZm9yIGksIChuYW1lLCBzY3JpcHQpIGluIGVudW1lcmF0ZShlbmdpbmVzKToKZGlmZiAtLWdpdCBhL2FwcC9tYWluLnB5IGIvYXBwL21haW4ucHkKaW5kZXggNTI5Zjk1Ny4uMjhhZDVjYiAxMDA2NDQKLS0tIGEvYXBwL21haW4ucHkKKysrIGIvYXBwL21haW4ucHkKQEAgLTksNiArOSw3IEBAIGltcG9ydCBwYW5kYXMgYXMgcGQKIGltcG9ydCBpbwogaW1wb3J0IG9zCiBpbXBvcnQgaG1hYworaW1wb3J0IG1hdGgKIGltcG9ydCB0ZW1wZmlsZQogCiBmcm9tIGFwcC5hbmFseXNpcy5wcm9maWxlciBpbXBvcnQgcHJvZmlsZV9kYXRhc2V0CkBAIC0yNSw2ICsyNiwyNCBAQCBhcHAgPSBGYXN0QVBJKAogICAgIGRlc2NyaXB0aW9uPSJTY2llbnRpZmljIG1pY3Jvc2VydmljZSBBUEkgZm9yIFJOQS1zZXEgZGlmZmVyZW50aWFsIGV4cHJlc3Npb24gKERFU2VxMiwgZWRnZVIsIGxpbW1hLXZvb20pLiIKICkKIAorCitkZWYgX2pzb25fc2FmZSh2YWx1ZSk6CisgICAgIiIiUmVwbGFjZSBOYU4vSW5maW5pdHkgd2l0aCBudWxsIHNvIFN0YXJsZXR0ZSBjYW4gcmVuZGVyIHN0cmljdCBKU09OLgorCisgICAgREVTZXEyL2VkZ2VSL2xpbW1hIGxlZ2l0aW1hdGVseSByZXR1cm4gTkEgZm9yIGFsbC16ZXJvIG9yIGZpbHRlcmVkIGdlbmVzLgorICAgIHBhbmRhcyB0dXJucyB0aG9zZSBpbnRvIE5hTiAoYW5kIG9jY2FzaW9uYWxseSArLy1JbmYgZm9yIGxvZzJGQyksIHdoaWNoIGFyZQorICAgIHZhbGlkIFB5dGhvbiBmbG9hdHMgYnV0IGludmFsaWQgSlNPTi4gRmFzdEFQSS9TdGFybGV0dGUgdGhlbiByYWlzZXMKKyAgICAiT3V0IG9mIHJhbmdlIGZsb2F0IHZhbHVlcyBhcmUgbm90IEpTT04gY29tcGxpYW50IiBhbmQgdGhlIHdob2xlIHJlc3VsdHMKKyAgICByZXF1ZXN0IGZhaWxzIHdpdGggYSA1MDAuIEVtaXR0aW5nIG51bGwgaW5zdGVhZCBsZXRzIHRoZSBVSSBza2lwIHRoZSByb3cuCisgICAgIiIiCisgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZmxvYXQpOgorICAgICAgICByZXR1cm4gdmFsdWUgaWYgbWF0aC5pc2Zpbml0ZSh2YWx1ZSkgZWxzZSBOb25lCisgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCk6CisgICAgICAgIHJldHVybiB7a2V5OiBfanNvbl9zYWZlKGl0ZW0pIGZvciBrZXksIGl0ZW0gaW4gdmFsdWUuaXRlbXMoKX0KKyAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAobGlzdCwgdHVwbGUpKToKKyAgICAgICAgcmV0dXJuIFtfanNvbl9zYWZlKGl0ZW0pIGZvciBpdGVtIGluIHZhbHVlXQorICAgIHJldHVybiB2YWx1ZQorCiBERUZBVUxUX1dPUktFUl9UT0tFTiA9ICJkZXYtdG9rZW4tMTIzIgogQklPSU5TSUdIVF9FTlYgPSBvcy5lbnZpcm9uLmdldCgiQklPSU5TSUdIVF9FTlYiLCAiZGV2ZWxvcG1lbnQiKQogV09SS0VSX1RPS0VOID0gb3MuZW52aXJvbi5nZXQoIkJJT0lOU0lHSFRfV09SS0VSX1RPS0VOIiwgREVGQVVMVF9XT1JLRVJfVE9LRU4pCkBAIC0xMjYsNyArMTQ1LDcgQEAgZGVmIGdldF9qb2Jfc3RhdHVzKGpvYl9pZDogc3RyKToKICAgICBpZiBub3Qgam9iOgogICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTQwNCwgZGV0YWlsPWYiSm9iIElEICd7am9iX2lkfScgbm90IGZvdW5kLiIpCiAgICAgCi0gICAgcmV0dXJuIHsKKyAgICByZXR1cm4gX2pzb25fc2FmZSh7CiAgICAgICAgICJqb2JfaWQiOiBqb2JbImpvYl9pZCJdLAogICAgICAgICAic3RhdGUiOiBqb2JbInN0YXRlIl0sCiAgICAgICAgICJwcm9ncmVzcyI6IGpvYlsicHJvZ3Jlc3MiXSwKQEAgLTEzNiw3ICsxNTUsNyBAQCBkZWYgZ2V0X2pvYl9zdGF0dXMoam9iX2lkOiBzdHIpOgogICAgICAgICAiY3JlYXRlZF9hdCI6IGpvYlsiY3JlYXRlZF9hdCJdLAogICAgICAgICAidXBkYXRlZF9hdCI6IGpvYlsidXBkYXRlZF9hdCJdLAogICAgICAgICAicWNfZGF0YSI6IGpvYi5nZXQoInFjX2RhdGEiKQotICAgIH0KKyAgICB9KQogCiBAYXBwLmdldCgiL2FwaS9hbmFseXNpcy97am9iX2lkfS9yZXN1bHRzIikKIGRlZiBnZXRfam9iX3Jlc3VsdHMoam9iX2lkOiBzdHIpOgpAQCAtMTQ3LDEyICsxNjYsMTIgQEAgZGVmIGdldF9qb2JfcmVzdWx0cyhqb2JfaWQ6IHN0cik6CiAgICAgaWYgam9iWyJzdGF0ZSJdICE9IEpvYlN0YXRlLkNPTVBMRVRFRDoKICAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbihzdGF0dXNfY29kZT00MDAsIGRldGFpbD1mIkpvYiAne2pvYl9pZH0nIGlzIG5vdCBjb21wbGV0ZSB5ZXQuIEN1cnJlbnQgc3RhdGU6IHtqb2JbJ3N0YXRlJ119IikKICAgICAKLSAgICByZXR1cm4geworICAgIHJldHVybiBfanNvbl9zYWZlKHsKICAgICAgICAgImpvYl9pZCI6IGpvYl9pZCwKICAgICAgICAgInN0YXR1cyI6ICJzdWNjZXNzIiwKICAgICAgICAgIndhcm5pbmdzIjogam9iWyJ3YXJuaW5ncyJdLAogICAgICAgICAicmVzdWx0cyI6IGpvYlsicmVzdWx0cyJdCi0gICAgfQorICAgIH0pCiAKIEBhcHAucG9zdCgiL2FwaS9hbmFseXNpcy97am9iX2lkfS9jYW5jZWwiKQogZGVmIGNhbmNlbF9qb2Ioam9iX2lkOiBzdHIpOgpkaWZmIC0tZ2l0IGEvd29ya2Vycy9jb2xhYl93b3JrZXIucHkgYi93b3JrZXJzL2NvbGFiX3dvcmtlci5weQppbmRleCBmZTVhMjY1Li5iMGQwYjZkIDEwMDY0NAotLS0gYS93b3JrZXJzL2NvbGFiX3dvcmtlci5weQorKysgYi93b3JrZXJzL2NvbGFiX3dvcmtlci5weQpAQCAtMSw0ICsxLDUgQEAKIGltcG9ydCBvcworaW1wb3J0IHJlCiBpbXBvcnQgc3lzCiBpbXBvcnQgdGltZQogaW1wb3J0IHV1aWQKQEAgLTI0LDYgKzI1LDQxIEBAIEhFQURFUlMgPSB7CiBkZWYgcHJpbnRfbG9nKG1zZzogc3RyKToKICAgICBwcmludChmIlt7dGltZS5zdHJmdGltZSgnJVktJW0tJWQgJUg6JU06JVMnKX1dIFt7V09SS0VSX0lEfV0ge21zZ30iLCBmbHVzaD1UcnVlKQogCisKK2RlZiBfbm9ybWFsaXNlX2Rlc2lnbihwYXJhbXM6IGRpY3QsIG1ldGFfcGF0aDogc3RyKSAtPiBkaWN0OgorICAgICIiIk1hdGNoIGRlc2lnbi9jb250cmFzdCBmYWN0b3IgbmFtZXMgdG8gdGhlIGNsZWFuZWQgbWV0YWRhdGEgY29sdW1ucy4KKworICAgIFRoZSBBUEkgc3RvcmVzIHRoZSBncm91cGluZyBjb2x1bW4gYXMgYGNvbmRpdGlvbmAgaW4gdGhlIC5jbGVhbi5jc3YuIE9sZGVyCisgICAgY2FsbGVycyBzdGlsbCBzZW5kIHRoZSBvcmlnaW5hbCBuYW1lIChlLmcuIGB+IEdyb3VwYCk7IHJld3JpdGUgaXQgc28gdGhlIFIKKyAgICBlbmdpbmVzIGRvIG5vdCBmYWlsIHdpdGggIm9iamVjdCAnR3JvdXAnIG5vdCBmb3VuZCIuCisgICAgIiIiCisgICAgdHJ5OgorICAgICAgICBjb2x1bW5zID0gbGlzdChwZC5yZWFkX2NzdihtZXRhX3BhdGgsIG5yb3dzPTApLmNvbHVtbnMpCisgICAgZXhjZXB0IEV4Y2VwdGlvbjoKKyAgICAgICAgcmV0dXJuIHBhcmFtcworICAgIHRhcmdldCA9ICJjb25kaXRpb24iIGlmICJjb25kaXRpb24iIGluIGNvbHVtbnMgZWxzZSBuZXh0KAorICAgICAgICAoYyBmb3IgYyBpbiBjb2x1bW5zIGlmIGMubG93ZXIoKSBub3QgaW4gKCJzYW1wbGUiLCAic2FtcGxlX2lkIiwgImlkIikpLCBOb25lCisgICAgKQorICAgIGlmIG5vdCB0YXJnZXQ6CisgICAgICAgIHJldHVybiBwYXJhbXMKKworICAgIHBhcmFtcyA9IGRpY3QocGFyYW1zIG9yIHt9KQorICAgIGRlc2lnbiA9IHN0cihwYXJhbXMuZ2V0KCJkZXNpZ24iKSBvciBmIn4ge3RhcmdldH0iKQorICAgIHJocyA9IGRlc2lnbi5zcGxpdCgifiIsIDEpWzFdIGlmICJ+IiBpbiBkZXNpZ24gZWxzZSBkZXNpZ24KKyAgICByaHMgPSByZS5zdWIoCisgICAgICAgIHIiW0EtWmEtel8uXVtBLVphLXowLTlfLl0qIiwKKyAgICAgICAgbGFtYmRhIG1hdGNoOiBtYXRjaC5ncm91cCgwKSBpZiBtYXRjaC5ncm91cCgwKSBpbiBjb2x1bW5zIGVsc2UgdGFyZ2V0LAorICAgICAgICByaHMsCisgICAgKQorICAgIHBhcmFtc1siZGVzaWduIl0gPSAifiAiICsgcmhzLnN0cmlwKCkKKworICAgIGNvbnRyYXN0ID0gcGFyYW1zLmdldCgiY29udHJhc3QiKQorICAgIGlmIGlzaW5zdGFuY2UoY29udHJhc3QsIHN0cikgYW5kIGNvbnRyYXN0OgorICAgICAgICBmYWN0b3IsIHNlcCwgcmVzdCA9IGNvbnRyYXN0LnBhcnRpdGlvbigiXyIpCisgICAgICAgIGlmIHNlcCBhbmQgZmFjdG9yIGFuZCBmYWN0b3Igbm90IGluIGNvbHVtbnM6CisgICAgICAgICAgICBwYXJhbXNbImNvbnRyYXN0Il0gPSBmInt0YXJnZXR9XyIgKyByZXN0CisgICAgcmV0dXJuIHBhcmFtcworCiBkZWYgZ2V0X3Nlc3Npb24oKToKICAgICBzID0gcmVxdWVzdHMuU2Vzc2lvbigpCiAgICAgcmV0cnkgPSBSZXRyeSgKQEAgLTEwMSw3ICsxMzcsOSBAQCBkZWYgcHJvY2Vzc19qb2Ioam9iX2lkOiBzdHIsIHBhcmFtczogZGljdCk6CiAgICAgICAgIHJfbWV0YS5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICAgd2l0aCBvcGVuKG1ldGFfcGF0aCwgIndiIikgYXMgZjoKICAgICAgICAgICAgIGYud3JpdGUocl9tZXRhLmNvbnRlbnQpCi0gICAgICAgICAgICAKKworICAgICAgICBwYXJhbXMgPSBfbm9ybWFsaXNlX2Rlc2lnbihwYXJhbXMsIG1ldGFfcGF0aCkKKwogICAgICAgICAjIExvY2F0ZSBSIHNjcmlwdHMKICAgICAgICAgYmFzZV9kaXIgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQogICAgICAgICBkZXNlcTJfc2NyaXB0ID0gb3MucGF0aC5qb2luKGJhc2VfZGlyLCAiYXBwIiwgImRlc2VxMiIsICJkZXNlcTJfc2NyaXB0LlIiKQo="
main_py = os.path.join(WORKDIR, "app", "main.py")
if "_json_safe" not in open(main_py, encoding="utf-8").read():
    patch_path = "/tmp/backend-hotfix.patch"
    with open(patch_path, "wb") as fh:
        fh.write(base64.b64decode(HOTFIX_B64))
    check = subprocess.run(["git", "-C", WORKDIR, "apply", "--check", patch_path],
                           capture_output=True, text=True)
    if check.returncode == 0:
        subprocess.run(["git", "-C", WORKDIR, "apply", patch_path], check=True)
        print("Backend hotfix applied: NA/Inf statistics are now JSON-safe.")
    else:
        print("Hotfix skipped (already present or context changed):", check.stderr.strip())
else:
    print("Backend hotfix already present.")


## STEP 9 — Pre-flight check

Verifies **everything** before starting the server: `Rscript`, all four R packages, every Python import, and the backend files. This is the cell that tells you whether STEP 2–8 actually worked.

In [ ]:
# STEP 9 — pre-flight
import os, shutil, subprocess

ok = True

def line(label, passed, detail=""):
    global ok
    ok = ok and passed
    print(f"{'OK  ' if passed else 'FAIL'}  {label:<26} {detail}")

rscript = shutil.which("Rscript")
line("Rscript on PATH", rscript is not None, rscript or "not found")

if rscript:
    ver = subprocess.run(["Rscript", "-e", "cat(R.version.string)"],
                         capture_output=True, text=True).stdout.strip()
    line("R version", bool(ver), ver)

    rcode = ('for (p in c("DESeq2","edgeR","limma","argparse")) '
             'cat(p, requireNamespace(p, quietly=TRUE), "\\n")')
    out = subprocess.run(["Rscript", "-e", rcode], capture_output=True, text=True).stdout
    for ln in out.strip().splitlines():
        parts = ln.split()
        if len(parts) == 2:
            line("R package " + parts[0], parts[1].upper() == "TRUE")

for mod in ["fastapi", "uvicorn", "pandas", "numpy", "multipart",
            "pydantic", "sklearn", "scipy", "httpx", "requests"]:
    try:
        __import__(mod)
        line("python " + mod, True)
    except Exception as e:
        line("python " + mod, False, str(e)[:60])

for f in ["app/main.py", "workers/colab_worker.py"]:
    line("file " + f, os.path.exists(os.path.join(WORKDIR, f)))

print()
if ok:
    print("ALL CHECKS PASSED — continue to STEP 10.")
else:
    print("SOMETHING IS MISSING — fix the FAIL lines above first.")
    print("Re-run STEP 2–7 for the failed item, then run this cell again.")

## STEP 10 — Start the backend

Starts FastAPI on `localhost:8000` with the Colab runtime and permissive CORS (needed so your deployed site can call the tunnel). Any previous backend process is stopped first, so this cell is safe to re-run.

In [ ]:
# STEP 10 — start the FastAPI backend
import os, subprocess, sys, time
import requests

subprocess.run(["pkill", "-f", "uvicorn app.main:app"], check=False)
time.sleep(1)

env = dict(
    os.environ,
    BIOINSIGHT_WORKER_TOKEN=WORKER_TOKEN,
    BIOINSIGHT_DB_PATH=DB_PATH,
    BIOINSIGHT_ALLOWED_ORIGINS="*",
    BIOINSIGHT_RUNTIME="colab",
    BIOINSIGHT_ENV="development",
    PYTHONUNBUFFERED="1",
)

log = open(BACKEND_LOG, "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=WORKDIR, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
)

print("Waiting for http://localhost:8000/health ...")
healthy = False
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        if r.ok:
            print("BACKEND UP:", r.json())
            healthy = True
            break
    except Exception:
        pass

if not healthy:
    print("\nBACKEND DID NOT START. Last 40 log lines:\n")
    print(open(BACKEND_LOG).read()[-4000:])
else:
    print(f"PID {proc.pid} · log {BACKEND_LOG}")

## STEP 11 — Open the public URL  ⬅ **THE URL YOU NEED**

Creates a temporary Cloudflare tunnel (no account needed), waits for the `https://….trycloudflare.com` address, **saves it to `/content/BACKEND_URL.txt`**, and prints it in a box you cannot miss.

> If this cell says *"No tunnel URL"*, wait 20 seconds and run it again. The troubleshooting table is at the end.

In [ ]:
# STEP 11 — public HTTPS tunnel (the URL comes from here)
import os, re, stat, subprocess, time, urllib.request
import requests
from IPython.display import display, HTML

PUBLIC_BACKEND_URL = (MANUAL_BACKEND_URL or "").strip()
CF_BIN = "/content/cloudflared"

def tail(path, n=40):
    try:
        return "\n".join(open(path).read().splitlines()[-n:])
    except Exception:
        return "(no log yet)"

if not PUBLIC_BACKEND_URL:
    # the backend must be running first
    try:
        requests.get("http://localhost:8000/health", timeout=3).raise_for_status()
    except Exception as e:
        print("localhost:8000 is not healthy — run STEP 10 first.")
        print("  ", e)

    subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
    time.sleep(1)

    if not os.path.exists(CF_BIN):
        print("Downloading cloudflared ...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CF_BIN,
        )
        os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)

    print("Starting the tunnel ...")
    tlog = open(TUNNEL_LOG, "w")
    subprocess.Popen(
        [CF_BIN, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True,
    )

    for _ in range(120):
        time.sleep(1)
        try:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
        except Exception:
            m = None
        if m:
            PUBLIC_BACKEND_URL = m.group(0)
            break

if not PUBLIC_BACKEND_URL:
    print("\nNO TUNNEL URL. Last cloudflared log lines:\n")
    print(tail(TUNNEL_LOG))
else:
    open(URL_FILE, "w").write(PUBLIC_BACKEND_URL)

    print("\nChecking the URL from outside Colab ...")
    public_ok = False
    for _ in range(10):
        try:
            r = requests.get(PUBLIC_BACKEND_URL + "/health", timeout=15)
            if r.ok:
                print("PUBLIC HEALTH:", r.json())
                public_ok = True
                break
        except Exception:
            time.sleep(2)

    print("\n" + "=" * 78)
    print("  COPY THIS URL")
    print("")
    print("  " + PUBLIC_BACKEND_URL)
    print("=" * 78)
    print("\nSaved to", URL_FILE, "— re-print it any time with STEP 14.")
    print("\nPaste it into the website (any one of these):")
    print("  live site :", SITE_URL + "/?backend=" + PUBLIC_BACKEND_URL)
    print("  local dev :", LOCAL_SITE_URL + "/?backend=" + PUBLIC_BACKEND_URL)
    print("  manually  : navbar → Backend → paste → Test & Save")

    if not public_ok:
        print("\nWARNING: URL created but the public health check failed.")
        print("Give it 10–20 s, then run STEP 14. If it still fails, see the troubleshooting table.")

    display(HTML(f"""
      <div style="border:2px solid #0284c7;border-radius:12px;padding:16px;margin:12px 0;
                  background:#f0f9ff;font-family:monospace">
        <div style="font-size:13px;color:#0369a1;margin-bottom:6px">
          PUBLIC BACKEND URL — copy this
        </div>
        <div style="font-size:20px;font-weight:bold;word-break:break-all">
          {PUBLIC_BACKEND_URL}
        </div>
      </div>
      <div style="font-family:sans-serif;font-size:14px">
        Open the site with this backend:
        <a href="{SITE_URL}/?backend={PUBLIC_BACKEND_URL}" target="_blank">live site</a> ·
        <a href="{LOCAL_SITE_URL}/?backend={PUBLIC_BACKEND_URL}" target="_blank">local dev</a>
      </div>
    """))

## STEP 12 — Start the worker

The worker polls the backend and runs the three R engines. It runs **in the background**, so you can keep using the notebook. It talks to `localhost:8000`, not the tunnel, which is faster and less fragile.

You should see `WORKER REGISTERED`.

In [ ]:
# STEP 12 — start the headless worker (background)
import os, subprocess, sys, time

subprocess.run(["pkill", "-f", "workers/colab_worker.py"], check=False)
time.sleep(1)

env = dict(
    os.environ,
    BIOINSIGHT_BACKEND_URL="http://localhost:8000",
    BIOINSIGHT_WORKER_TOKEN=WORKER_TOKEN,
    PYTHONUNBUFFERED="1",
)

wlog = open(WORKER_LOG, "w")
proc = subprocess.Popen(
    [sys.executable, "workers/colab_worker.py"],
    cwd=WORKDIR, env=env, stdout=wlog, stderr=subprocess.STDOUT, start_new_session=True,
)
time.sleep(6)

tail = open(WORKER_LOG).read()
print("worker PID:", proc.pid, "· log:", WORKER_LOG)
if "Successfully registered" in tail:
    print("WORKER REGISTERED — jobs submitted from the website will now run.")
else:
    print("WORKER NOT CONFIRMED YET. Log so far:\n")
    print(tail[-2000:])
    print("\nMost common cause: the backend is not running (re-run STEP 10).")

## STEP 13 — End-to-end test (proves the whole system works)

Downloads the demo dataset (GSE52778), submits it through the **same API the website uses**, waits for the worker, and prints the gene counts from all three engines.

The backend renames the grouping column to `condition` before running R, so the request below sends `design = "~ condition"` even though the CSV column is `group`.

**If this cell ends with "END-TO-END OK", the website will work.** No further debugging needed.

In [ ]:
# STEP 13 — submit a real job and wait for the result
import json, os, re, time
import pandas as pd
import requests

BASE = "http://localhost:8000"
COUNTS = os.path.join(DATA_DIR, "counts.csv")
META = os.path.join(DATA_DIR, "metadata.csv")

# 1) demo data, with a synthetic fallback so this test never depends on the network
try:
    for name, dest in [("demo_counts.csv", COUNTS), ("demo_metadata.csv", META)]:
        url = f"https://raw.githubusercontent.com/{FRONTEND_REPO}/{FRONTEND_BRANCH}/frontend/public/data/{name}"
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        open(dest, "wb").write(r.content)
    print("Demo data downloaded (GSE52778).")
except Exception as e:
    print("Demo download failed — generating a synthetic dataset instead:", e)
    import numpy as np
    rng = np.random.default_rng(7)
    genes = [f"GENE{i:04d}" for i in range(300)]
    samples = [f"S{i}" for i in range(1, 9)]
    counts = rng.negative_binomial(5, 0.01, size=(len(genes), len(samples)))
    counts[100:150, 4:] *= 4
    pd.DataFrame(counts, index=genes, columns=samples).to_csv(COUNTS)
    pd.DataFrame({"sample": samples, "group": ["CONTROL"] * 4 + ["TREATED"] * 4}).to_csv(META, index=False)

meta = pd.read_csv(META)
group_col = [c for c in meta.columns if c.lower() not in ("sample", "sample_id", "id")][0]
groups = list(dict.fromkeys(meta[group_col].astype(str)))
control = next((g for g in groups if re.search(r"control|ctrl|untreated|baseline|vehicle|wt", g, re.I)), groups[0])
treated = next((g for g in groups if g != control), groups[-1])

# The backend renames whichever grouping column it resolves (here `group`)
# to `condition` before the R engines run. The design/contrast must therefore
# use `condition`, not the original column name.
params = {
    "control_group": control,
    "treated_group": treated,
    "condition_col": group_col,
    "design": "~ condition",
}
if re.fullmatch(r"[A-Za-z0-9.]+", str(control)) and re.fullmatch(r"[A-Za-z0-9.]+", str(treated)):
    params["contrast"] = f"condition_{treated}_vs_{control}"

print(f"Submitting: control={control!r}  treated={treated!r}  design=~ condition")

with open(COUNTS, "rb") as cf, open(META, "rb") as mf:
    r = requests.post(
        f"{BASE}/api/analysis",
        files={"counts_file": ("counts.csv", cf, "text/csv"),
               "metadata_file": ("metadata.csv", mf, "text/csv")},
        data={"params": json.dumps(params)},
        timeout=120,
    )

if r.status_code != 202:
    raise SystemExit(f"Submit failed: HTTP {r.status_code}\n{r.text[:1000]}")

job = r.json()["job_id"]
print("job_id:", job)

status, last = None, None
for _ in range(180):                      # up to 15 minutes
    time.sleep(5)
    status = requests.get(f"{BASE}/api/analysis/{job}", timeout=30).json()
    state = f"{status['state']} {status.get('progress', 0)}% — {status.get('message', '')}"
    if state != last:
        print("  ", state)
        last = state
    if status["state"] in ("COMPLETED", "FAILED", "CANCELLED"):
        break

if not status:
    raise SystemExit("Could not read the job status — is the backend still running? (STEP 15)")

if status["state"] != "COMPLETED":
    print("\nJOB ENDED AS", status["state"])
    print("error:", status.get("error"))
    print("\n--- worker log (last lines) ---")
    print(open(WORKER_LOG).read()[-3000:])
    print("\n--- backend log (last lines) ---")
    print(open(BACKEND_LOG).read()[-3000:])
else:
    print("\nJOB COMPLETED. warnings:", status.get("warnings"))
    qc = status.get("qc_data") or {}
    if qc:
        print("QC samples       :", qc.get("sampleNames"))
        print("QC genes retained:", qc.get("genesRetained"), "/", qc.get("totalGenes"))

    results = requests.get(f"{BASE}/api/analysis/{job}/results", timeout=120).json()["results"]
    for engine in ("deseq2", "edger", "limma"):
        rows = results.get(engine) or []
        sig = [x for x in rows if x.get("padj") is not None and x["padj"] < 0.05]
        print(f"  {engine:<6} {len(rows):>4} genes tested · {len(sig):>3} with padj < 0.05")

    top = sorted([x for x in (results.get("deseq2") or [])
                  if x.get("padj") is not None and x.get("log2FoldChange") is not None],
                 key=lambda x: x["padj"])[:3]
    if top:
        print("\nTop 3 genes by padj (DESeq2):")
        for x in top:
            print(f"   {x['Gene']:<14} log2FC={x['log2FoldChange']:+.2f}   padj={x['padj']:.3g}")

    if all(results.get(e) for e in ("deseq2", "edger", "limma")):
        print("\nEND-TO-END OK — all three engines returned results.")
        print("Next: STEP 14 → copy the URL → paste into the site → Upload → QC → Consensus → Pathways → Export.")
    else:
        print("\nPARTIAL: at least one engine failed. Check warnings and the worker log.")

## STEP 14 — Your URL (re-print any time)

Run this whenever you need the address again, for example after scrolling too far up.

In [ ]:
# STEP 14 — re-print the public URL and the exact paste locations
import os
from IPython.display import display, HTML

url = ""
if os.path.exists(URL_FILE):
    url = open(URL_FILE).read().strip()

if not url:
    print("No URL saved yet — run STEP 11 first.")
else:
    print("=" * 78)
    print("  YOUR BACKEND URL")
    print("")
    print("  " + url)
    print("=" * 78)
    print()
    print("Where to paste it — pick ONE:")
    print()
    print("  A) Live site, no redeploy:")
    print("     " + SITE_URL + "/?backend=" + url)
    print()
    print("  B) Local dev server (this machine):")
    print("     " + LOCAL_SITE_URL + "/?backend=" + url)
    print()
    print("  C) Manually: open the site → navbar → Backend → paste → Test & Save")
    print("     Expected result: {\"status\": \"healthy\"}")
    print()
    print("Then: Upload → Load Demo Dataset → Continue → QC → Consensus → Pathways → Export")

    display(HTML(f"""
      <div style="font-family:sans-serif">
        <p style="margin:0 0 6px 0;color:#0369a1;font-size:13px">PUBLIC BACKEND URL</p>
        <p style="font-family:monospace;font-size:20px;font-weight:bold;word-break:break-all;
                  margin:0 0 12px 0">{url}</p>
        <p style="margin:0">
          <a href="{SITE_URL}/?backend={url}" target="_blank">Open the live site with this backend</a>
          &nbsp;·&nbsp;
          <a href="{LOCAL_SITE_URL}/?backend={url}" target="_blank">Open local dev</a>
        </p>
      </div>
    """))

## STEP 15 — Logs and diagnostics

A snapshot of all three processes and their last log lines. Re-run it whenever something looks wrong.

In [ ]:
# STEP 15 — status + log snapshot (non-blocking)
import os, subprocess

print("PROCESSES")
out = subprocess.run(
    ["pgrep", "-af", "uvicorn app.main:app|workers/colab_worker.py|cloudflared tunnel"],
    capture_output=True, text=True).stdout.strip()
print(out if out else "  (nothing running — re-run STEP 10 and 12)")

try:
    import requests
    print("\nlocal  /health:", requests.get("http://localhost:8000/health", timeout=5).json())
    if os.path.exists(URL_FILE):
        u = open(URL_FILE).read().strip()
        print("public /health:", requests.get(u + "/health", timeout=15).json())
except Exception as e:
    print("health check failed:", e)

for name, path, n in [("BACKEND", BACKEND_LOG, 25),
                      ("WORKER", WORKER_LOG, 25),
                      ("TUNNEL", TUNNEL_LOG, 12)]:
    print("\n" + "-" * 28, name, "-" * 28)
    try:
        print("\n".join(open(path).read().splitlines()[-n:]))
    except Exception as e:
        print("no log:", e)

## STEP 16 — *Optional:* thesis validation outputs

Runs the three engines directly on the demo data (no web UI, no worker), saves one CSV per engine, draws a volcano plot per engine, and gives you a downloadable ZIP. Use this for the thesis results section.

You can run this while the backend is running — it uses separate files and does not touch the job database.

In [ ]:
# STEP 16 — direct R-engine validation outputs + ZIP
import os, re, shutil, subprocess, time
import pandas as pd

OUT = "/content/bioinsight_validation"
os.makedirs(OUT, exist_ok=True)
COUNTS = os.path.join(DATA_DIR, "counts.csv")
META = os.path.join(DATA_DIR, "metadata.csv")

if not (os.path.exists(COUNTS) and os.path.exists(META)):
    raise SystemExit("Demo data missing — run STEP 13 once (it downloads the dataset).")

meta = pd.read_csv(META)
group_col = [c for c in meta.columns if c.lower() not in ("sample", "sample_id", "id")][0]
groups = list(dict.fromkeys(meta[group_col].astype(str)))
control = next((g for g in groups if re.search(r"control|ctrl|untreated|baseline|vehicle|wt", g, re.I)), groups[0])
treated = next((g for g in groups if g != control), groups[-1])
contrast = ""
if re.fullmatch(r"[A-Za-z0-9.]+", str(control)) and re.fullmatch(r"[A-Za-z0-9.]+", str(treated)):
    contrast = f"{group_col}_{treated}_vs_{control}"

scripts = {
    "DESeq2": "app/deseq2/deseq2_script.R",
    "edgeR":  "app/edger/edger_script.R",
    "limma":  "app/limma/limma_script.R",
}

outputs = {}
for name, rel in scripts.items():
    out = os.path.join(OUT, f"{name}_results.csv")
    cmd = ["Rscript", os.path.join(WORKDIR, rel),
           "--counts", COUNTS, "--meta", META, "--out", out,
           "--design", f"~ {group_col}", "--contrast", contrast]
    print(f"\n=== {name} ===")
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print("FAILED:\n", (p.stderr or p.stdout)[-1500:])
        continue
    df = pd.read_csv(out)
    outputs[name] = df
    sig = df[(df["padj"].notna()) & (df["padj"] < 0.05)]
    fc = sig[(sig["log2FoldChange"].abs() > 1)]
    print(f"  genes tested: {len(df)}  |  padj<0.05: {len(sig)}  |  padj<0.05 & |log2FC|>1: {len(fc)}")
    print(f"  saved {out}  ({time.time()-t0:.1f}s)")

# volcano plots
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

for name, df in outputs.items():
    d = df.dropna(subset=["padj"])
    if d.empty:
        continue
    x = d["log2FoldChange"]
    y = -np.log10(d["padj"].clip(lower=1e-300))
    sig = (d["padj"] < 0.05) & (d["log2FoldChange"].abs() > 1)
    plt.figure(figsize=(7, 5))
    plt.scatter(x[~sig], y[~sig], s=10, c="#94a3b8", alpha=0.7)
    plt.scatter(x[sig], y[sig], s=14, c="#dc2626", alpha=0.9)
    plt.axhline(-np.log10(0.05), ls="--", c="grey", lw=0.8)
    plt.axvline(1, ls="--", c="grey", lw=0.8)
    plt.axvline(-1, ls="--", c="grey", lw=0.8)
    plt.xlabel("log2 fold change")
    plt.ylabel("-log10 adjusted p")
    plt.title(f"{name}: {treated} vs {control}")
    plt.tight_layout()
    png = os.path.join(OUT, f"{name}_volcano.png")
    plt.savefig(png, dpi=150)
    plt.close()
    print("saved", png)

zip_path = shutil.make_archive("/content/bioinsight_validation", "zip", OUT)
print("\nZIP:", zip_path, f"({os.path.getsize(zip_path)} bytes)")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download it from the Colab file browser:", zip_path)

## STEP 17 — Stop / restart

Run the stop cell when you want a clean slate. Then re-run **STEP 10 → 11 → 12**.

Stop is also useful if you see `Address already in use` or a worker from an old run is picking up jobs.

In [ ]:
# STEP 17 — stop backend + tunnel + worker
import subprocess

for pattern in ("uvicorn app.main:app", "workers/colab_worker.py", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", pattern], check=False)

print("Stopped backend, worker and tunnel.")
print("The public URL is now dead. Re-run STEP 10 → 11 → 12 to get a new one.")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Rscript: command not found` | STEP 2 failed | Re-run STEP 2, then STEP 9 |
| `there is no package called 'argparse'` | STEP 3 skipped | Run STEP 3 |
| `there is no package called 'DESeq2'` / `edgeR` / `limma` | STEP 4–6 failed | Re-run the failing step, then STEP 9 |
| DESeq2 install takes forever | compiling from source, normal | wait; 10–25 min |
| STEP 7 `pip` fails / cell stops | `--upgrade` conflicts with Colab numpy/pandas | use the no-upgrade STEP 7 in this notebook; it keeps Colab's stack |
| `ModuleNotFoundError` when starting the backend | STEP 7 not finished | Re-run STEP 7, then STEP 10 |
| `Address already in use` | old backend still running | run STEP 17, then STEP 10 |
| STEP 11 says **No tunnel URL** | cloudflared slow / blocked | wait 20 s, run STEP 11 again; check STEP 15 tunnel log |
| URL works locally but public health fails | tunnel still warming up | wait, run STEP 14; retry STEP 11 |
| Worker says **Failed to register** | backend down, or token mismatch | re-run STEP 10, then STEP 12 |
| Job stays `QUEUED` forever | worker not running | run STEP 12, confirm "WORKER REGISTERED" |
| Job `FAILED`: `object 'condition' not found` or `object 'Group' not found` | design / grouping-column mismatch; the backend renames the grouping column to `condition` | send `condition_col=<real column>` and `design='~ condition'` (the current frontend and STEP 13 do) |
| Job `FAILED` with an R error | data or R package problem | read STEP 15 worker log; the message is the R error |
| Upload returns **422** | counts/metadata sample names do not match | ensure the same sample IDs in both files, ≥ 2 per group |
| Site says **Analysis Not Found** | browser lost the job ID (new tab / Colab restart) | redo Upload → QC; the job lives in Colab only |
| Colab disconnected | free runtime limit | re-run STEP 1, 9, 10, 11, 12 — the URL changes |
| Website cannot reach the URL | mixed content or old deployment | site and tunnel must both be `https`; or use the navbar Backend box |

### Good to know

- The tunnel URL is **temporary**. It changes every time you restart STEP 11.
- The job database is `/content/bioinsight_jobs.db` and disappears when the Colab runtime is recycled.
- `WORKER_TOKEN` is `dev-token-123`. Fine for a demo; change it before any public release (and set it identically in STEP 1 and the backend).
- Free Colab is enough for the demo (200 genes × 8 samples). The worker is CPU-only.

## Where exactly to paste the URL

### Option A — fastest (no redeploy)

Open:

```
<SITE_URL>/?backend=<YOUR-TUNNEL-URL>
```

For example:

```
https://bioinsight-360.vercel.app/?backend=https://random-words.trycloudflare.com
```

STEP 14 prints this full link and makes it clickable. The app reads `?backend=`, saves it, and uses it immediately.

### Option B — through the UI

1. Open the website.
2. Click **Backend** in the navbar.
3. Paste the `https://….trycloudflare.com` URL.
4. Click **Test & Save**. You should see `{"status": "healthy"}`.
5. Go to **Upload → Load Demo Dataset → Continue**.

### Option C — local dev server

If you are running the site on this machine:

```
http://localhost:3000/?backend=<YOUR-TUNNEL-URL>
```

or navbar → Backend → paste → Test & Save.

---

### After the URL is connected

```
Upload (demo GSE52778)
   → Continue
   → QC            (library sizes, PCA, correlation)
   → Consensus     (DESeq2 + edgeR + limma, ≥3/3 agreement)
   → Pathways      (g:Profiler enrichment)
   → Export        (CSV / report)
```

If STEP 13 printed **END-TO-END OK**, every one of those pages has real data behind it.